# 📘 Notebook 10: Spark SQL and Pipelines

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module C: Too Big for One Machine · Notebook 4 of 4 in this module · (10 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Run plain **SQL** on Spark, and show that it is the same thing as a DataFrame query
- Join tables in Spark and explain the two ways Spark does it: **broadcast** join and **shuffle** join
- Explain why a Python **user-defined function** is slow, and what to use in its place
- Assemble a **pipeline**: read, clean, enrich, summarise, with checks on the data at each stage
- Write the result as partitioned **Parquet**
- Decide, from measurements, **when a cluster engine is the right tool and when it is not**

> **Prerequisites:** Notebook 09 (Spark sessions, DataFrames, lazy evaluation, query plans). Notebooks 02 and 04 (joins, window functions, the warehouse). Notebook 05 (Parquet).
>
> 🔭 **Why this notebook matters:** this notebook closes two circles. The first: the SQL you learned in Module A for one small database runs, word for word, on an engine built for a thousand machines. That is the declarative idea of Notebook 01 at its full strength. The second: for nine notebooks we have been building towards heavier and heavier machinery, and it is time to ask plainly what all of it costs. The most expensive mistake in this field is not failing to scale. It is paying for a cluster to do a job that a laptop would have finished sooner.

> 📦 **Libraries used in this notebook.**
>
> - **PySpark**, introduced in Notebook 09.
> - **SQLite**, built into Python, as the single-machine database to compare against.
>
> The cell below installs what is needed. In Google Colab it is already present and the cell finishes in a moment.

In [ ]:
%pip install -q pyspark

In [ ]:
import csv
import datetime
import os
import random
import sqlite3
import tempfile
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = (SparkSession.builder
         .master("local[*]")
         .appName("bookshop")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is running on", spark.sparkContext.defaultParallelism, "cores")

The four tables of the bookshop, as they were in Module A, written as CSV files. A real company would receive them like this, as a nightly export from its operational database.

In [ ]:
def make_shop(customers=1000, products=200, orders=10000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    customer_rows = [(i, f"Customer {i}", rng.choice(cities), 1950 + rng.randint(0, 55)) for i in range(1, customers + 1)]
    product_rows = [(i, f"Book {i}", rng.choice(categories), round(rng.uniform(5, 60), 2)) for i in range(1, products + 1)]
    order_rows, item_rows = [], []
    first_day = datetime.date(2023, 1, 1)
    for order_id in range(1, orders + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        order_rows.append((order_id, rng.randint(1, customers), day.isoformat()))
        for line in range(rng.randint(1, 4)):
            item_rows.append((len(item_rows) + 1, order_id, rng.randint(1, products), rng.randint(1, 3)))
    return customer_rows, product_rows, order_rows, item_rows

folder = tempfile.mkdtemp()

def write_csv(name, header, rows):
    path = os.path.join(folder, name + ".csv")
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)
    return path

customer_rows, product_rows, order_rows, item_rows = make_shop(customers=5000, orders=100000)
paths = {
    "customers":   write_csv("customers", ["customer_id", "name", "city", "birth_year"], customer_rows),
    "products":    write_csv("products", ["product_id", "title", "category", "price"], product_rows),
    "orders":      write_csv("orders", ["order_id", "customer_id", "order_date"], order_rows),
    "order_items": write_csv("order_items", ["item_id", "order_id", "product_id", "quantity"], item_rows),
}

tables = {}
for name, path in paths.items():
    tables[name] = spark.read.csv(path, header=True, inferSchema=True)
    tables[name].createOrReplaceTempView(name)                 # give the DataFrame a name that SQL can use
    print(f"{name:<12} {tables[name].count():>8,} rows   columns: {tables[name].columns}")

## 1. SQL on Spark

The line `createOrReplaceTempView` registered each DataFrame under a table name. From that moment `spark.sql(...)` accepts ordinary SQL about them.

Here is the query from Notebook 02 for revenue by category, exactly as it was written for SQLite.

In [ ]:
by_category = spark.sql("""
    SELECT p.category,
           SUM(i.quantity)                    AS copies_sold,
           ROUND(SUM(i.quantity * p.price))   AS revenue
    FROM order_items AS i
    JOIN products AS p ON p.product_id = i.product_id
    GROUP BY p.category
    ORDER BY revenue DESC
""")
by_category.show(4)

`spark.sql` returns a **DataFrame**, lazy like any other. SQL and the DataFrame methods of Notebook 09 are two ways of writing the same thing. To be convinced, write the query both ways and compare what Spark plans to do.

In [ ]:
in_sql = spark.sql("SELECT city, COUNT(*) AS customers FROM customers WHERE birth_year >= 1990 GROUP BY city")
with_methods = tables["customers"].filter(F.col("birth_year") >= 1990).groupBy("city").agg(F.count("*").alias("customers"))

def plan_text(frame):
    lines = frame._jdf.queryExecution().optimizedPlan().toString().splitlines()
    return [line.split("#")[0].strip() for line in lines]              # drop Spark's internal numbering of columns

print("Optimised plan from SQL:")
for line in plan_text(in_sql):
    print("  ", line)
print("\nSame steps from the DataFrame methods:", plan_text(in_sql) == plan_text(with_methods))
print("Same result:", sorted(in_sql.collect()) == sorted(with_methods.collect()))

Both are translated into the same plan, optimised by the same optimiser, and run by the same engine. Use whichever reads better. Analysts tend to prefer SQL. Programmers building pipelines tend to prefer the methods, which can be split into functions and tested.

The window functions of Notebook 02 work as well. The three best-selling books of each category:

In [ ]:
spark.sql("""
    SELECT category, title, revenue, position
    FROM (
        SELECT p.category, p.title,
               ROUND(SUM(i.quantity * p.price)) AS revenue,
               RANK() OVER (PARTITION BY p.category ORDER BY SUM(i.quantity * p.price) DESC) AS position
        FROM order_items AS i
        JOIN products AS p ON p.product_id = i.product_id
        GROUP BY p.category, p.title
    )
    WHERE position <= 3
    ORDER BY category, position
""").show(6)

A query written in 2024 for a file-sized database, running unchanged on an engine designed for clusters. Nothing in the text of a SQL query says how many machines it will run on. That is what "declarative" bought us in Notebook 01, and this is where the purchase pays off.

## 2. How Spark joins

In Notebook 02 we wrote a join by hand twice: a nested loop, and a hash join that first built a dictionary from the smaller table. In Notebook 08 we did it a third way, by shuffling both tables on the join key so that matching rows met at the same reducer. Spark uses the last two ideas, and chooses between them.

### Shuffle join
The general method. Both tables are shuffled by the join key, so that all rows with the same key arrive in the same partition, and are matched there. It works for tables of any size, and it shuffles **both** tables in full.

### Broadcast join
When one table is small, there is a much cheaper way. Send a complete copy of the small table to **every** worker. Each worker builds a dictionary from it and looks up the rows of its own partition of the big table, where they already are. The big table is never shuffled at all.

Spark chooses automatically, from its estimate of the size of each table. Let us see what it does with our 200 products.

In [ ]:
revenue_query = """
    SELECT p.category, SUM(i.quantity * p.price) AS revenue
    FROM order_items AS i JOIN products AS p ON p.product_id = i.product_id
    GROUP BY p.category
"""

def join_methods(frame):
    plan = frame._jdf.queryExecution().executedPlan().toString()
    return [name for name in ["BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin"] if name in plan]

print("Join chosen by Spark:", join_methods(spark.sql(revenue_query)))

A **broadcast** join: the products table is tiny, so it is copied to the workers. Now we forbid broadcasting, to see the alternative and what it costs.

In [ ]:
def time_query(sql, repeats=3):
    best = float("inf")
    for i in range(repeats):
        started = time.perf_counter()
        spark.sql(sql).collect()
        best = min(best, time.perf_counter() - started)
    return best

with_broadcast = time_query(revenue_query)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")          # never broadcast
print("Join chosen now:     ", join_methods(spark.sql(revenue_query)))
without_broadcast = time_query(revenue_query)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))     # back to the default, 10 MB

print(f"\nBroadcast join: {with_broadcast:.2f} s")
print(f"Shuffle join:   {without_broadcast:.2f} s")

The shuffle join had to move a quarter of a million order lines between partitions and sort them. The broadcast join moved two hundred products. On one machine, where "moving" data means copying it in memory, the difference is modest. On a real cluster, where a shuffle crosses the network, it is much larger.

| | One table is small | Both tables are large |
|---|---|---|
| Broadcast join | copies the small table to every worker, no shuffle | impossible: the "small" table would not fit in a worker's memory |
| Shuffle join | works, and shuffles the big table for nothing | the only method that works |
| Better choice | **broadcast** | **shuffle** |

This is the star schema of Notebook 04 showing its worth on a cluster. A huge fact table surrounded by small dimension tables is exactly the shape for which broadcast joins were made: the facts stay where they are, and the dimensions travel.

## 3. The cost of leaving the engine

Sometimes the built-in functions do not seem to cover what you need, and Spark lets you supply your own Python function, a **user-defined function** or UDF. It is convenient, and it has a price that Notebook 09 already hinted at.

Spark's engine runs on the Java virtual machine. A Python function cannot run there. For every row, the data must be handed across to a Python process, the function called, and the answer handed back. The optimiser, moreover, cannot see inside the function.

Let us put the category in capital letters two ways: with Spark's own `upper`, and with a Python UDF that does the same.

In [ ]:
items_and_products = spark.sql("""
    SELECT i.item_id, p.category FROM order_items AS i JOIN products AS p ON p.product_id = i.product_id
""").cache()
items_and_products.count()                                             # fill the cache, so that we time only the function

python_upper = F.udf(lambda text: text.upper(), StringType())

def time_frame(frame):
    started = time.perf_counter()
    frame.groupBy("shouted").count().collect()
    return time.perf_counter() - started

built_in = time_frame(items_and_products.select(F.upper("category").alias("shouted")))
with_udf = time_frame(items_and_products.select(python_upper("category").alias("shouted")))

print(f"Built-in function: {built_in:.2f} s")
print(f"Python UDF:        {with_udf:.2f} s   ({with_udf / built_in:.0f} times slower)")
items_and_products.unpersist()
print("(cache released)")

The same result, several times slower, for a function that does almost nothing. The time went on carrying a quarter of a million values out of the engine and back.

> ⚠️ **Common pitfalls**
>
> - Reaching for a Python UDF before looking for a built-in function. Spark has several hundred, for text, dates, numbers and arrays, and a combination of them covers nearly every need.
> - Joining two large tables on a key that is heavily **skewed**. As in Notebook 08, all the rows of the commonest key land in one partition, and that partition decides how long the job takes.
> - Calling `collect()` to "look at the data". It brings everything to one machine. Use `show()` or `limit(n)`.
> - Forgetting that every action re-runs the whole plan. A long pipeline with several actions and no caching does its expensive first steps several times.

## 4. A pipeline

### Intuition first
Real work with data is rarely one query. It is a sequence of stages that runs every night: take what arrived, throw out what is broken, attach what is needed from other tables, and produce the tables that people will actually look at. Such a sequence is a **pipeline**.

The data we have used so far has been perfectly clean, which real data never is. Let us spoil it realistically. The file below is a new batch of order lines, as it might arrive from a faulty export: some rows have no quantity, some have a negative one, some refer to a product that does not exist, and some arrive twice.

In [ ]:
rng = random.Random(42)
raw_rows = []
for item_id, order_id, product_id, quantity in item_rows[:60000]:
    roll = rng.random()
    if roll < 0.01:
        quantity = ""                                        # missing
    elif roll < 0.02:
        quantity = -quantity                                 # impossible
    elif roll < 0.03:
        product_id = 9000 + product_id                       # no such product
    raw_rows.append((item_id, order_id, product_id, quantity))
    if roll > 0.985:
        raw_rows.append((item_id, order_id, product_id, quantity))      # the same line sent twice

raw_path = write_csv("raw_items", ["item_id", "order_id", "product_id", "quantity"], raw_rows)
raw = spark.read.csv(raw_path, header=True, inferSchema=True)
print(f"{raw.count():,} rows arrived")

Each stage of the pipeline is a function that takes a DataFrame and returns a DataFrame. Small stages like these can be read, tested and reused one at a time.

In [ ]:
def clean(items):
    """Remove duplicates and rows that cannot be right."""
    return (items.dropDuplicates(["item_id"])
                 .filter(F.col("quantity").isNotNull())
                 .filter(F.col("quantity") > 0))

def enrich(items, products, orders, customers):
    """Attach the descriptions needed for analysis. Rows whose product is unknown are dropped by the inner join."""
    return (items.join(F.broadcast(products), "product_id")
                 .join(orders, "order_id")
                 .join(F.broadcast(customers.select("customer_id", "city")), "customer_id")
                 .withColumn("revenue", F.round(F.col("quantity") * F.col("price"), 2))
                 .withColumn("year", F.year("order_date"))
                 .withColumn("month", F.month("order_date"))
                 .select("item_id", "order_date", "year", "month", "city", "category", "title", "quantity", "revenue"))

def summarise(sales):
    """The table people will read: one row per month and category."""
    return (sales.groupBy("year", "month", "category")
                 .agg(F.sum("quantity").alias("copies"), F.round(F.sum("revenue"), 2).alias("revenue"))
                 .orderBy("year", "month", "category"))

cleaned = clean(raw)
sales = enrich(cleaned, tables["products"], tables["orders"], tables["customers"]).cache()
monthly = summarise(sales)

monthly.show(4)

`F.broadcast(...)` is a hint: we know these two tables are small, and tell Spark to broadcast them whatever its own estimate says.

Notice that the three function calls before `show` did no work. They assembled one long plan, and `show` ran it. Because the whole pipeline is a single plan, Spark optimises **across** our stages: a column that the last stage does not need is never read by the first.

### Checking the data
A pipeline that runs every night without anyone watching must check its own work. The simplest checks are counts: how many rows came in, how many were rejected at each stage, and does anything impossible remain?

In [ ]:
arrived = raw.count()
after_cleaning = cleaned.count()
after_enriching = sales.count()

print(f"Rows arrived:                 {arrived:>7,}")
print(f"Removed by cleaning:          {arrived - after_cleaning:>7,}   (duplicates, missing or impossible quantities)")
print(f"Removed for unknown products: {after_cleaning - after_enriching:>7,}")
print(f"Rows kept:                    {after_enriching:>7,}   ({after_enriching / arrived:.1%})")

checks = {
    "no duplicated item_id":      sales.count() == sales.select("item_id").distinct().count(),
    "no missing values":          sales.dropna().count() == sales.count(),
    "every quantity is positive": sales.filter(F.col("quantity") <= 0).count() == 0,
    "every revenue is positive":  sales.filter(F.col("revenue") <= 0).count() == 0,
}
for description, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {description}")

Two habits are on display. The pipeline **reports** how much it threw away. A night on which 3% of rows are rejected is ordinary. A night on which 60% are rejected means something upstream has broken, and somebody should be told before the morning's reports are wrong. And it **verifies** what it produces, since a wrong number in a report is worse than a missing report.

Rejecting bad rows silently, as our inner join did with unknown products, is acceptable only because we counted them. In production the rejected rows are usually written to a table of their own, so that someone can find out why they were bad.

### Writing the result
*The cells in this subsection write files with Spark, and are meant for Colab.*

The last stage stores the result where other tools can find it: as Parquet, partitioned by year and month, in the folder layout of Notebook 05.

In [ ]:
output_path = os.path.join(folder, "warehouse", "sales")

(sales.write
      .mode("overwrite")                       # running the pipeline again replaces the output: no duplicates
      .partitionBy("year", "month")
      .parquet(output_path))

for directory, subfolders, files in sorted(os.walk(output_path))[:4]:
    print(os.path.relpath(directory, folder).replace(os.sep, "/"), [name for name in files if name.endswith(".parquet")][:1])

In [ ]:
march = spark.read.parquet(output_path).filter((F.col("year") == 2024) & (F.col("month") == 3))
print(f"{march.count():,} order lines in March 2024")
march.groupBy("category").agg(F.round(F.sum("revenue")).alias("revenue")).orderBy(F.desc("revenue")).show(3)
march.explain()

In the plan, look for `PartitionFilters`. The conditions on `year` and `month` were used to choose which **folders** to open. The other twenty-three months were never read. The mode `overwrite` matters as much: a pipeline will be run again after a failure, and running it twice must not double the data. An operation that can safely be repeated is called **idempotent**, the same property that let MapReduce re-run a failed chunk.

In [ ]:
sales.unpersist()
print("(cache released)")

## 5. Did we need a cluster engine?

Now the question that this module has been postponing. Spark took several seconds to start. Every query in this notebook took a noticeable fraction of a second. The SQLite queries of Module A, on the same data, were finished almost before the key was released. What have we gained?

Let us measure properly. The same query, a grouped total, on tables of twenty thousand, two hundred thousand and two million rows, in SQLite and in Spark. To be fair to both, the data is loaded and held in memory first, and only the query is timed.

In [ ]:
def write_lines(path, lines, seed=0):
    rng = random.Random(seed)
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["line_id", "category", "quantity", "revenue"])
        for line_id in range(lines):
            quantity = rng.randint(1, 3)
            writer.writerow([line_id, rng.choice(categories), quantity, round(quantity * rng.uniform(5, 60), 2)])

results = []
for lines in [20_000, 200_000, 2_000_000]:
    path = os.path.join(folder, f"lines_{lines}.csv")
    write_lines(path, lines)

    db = sqlite3.connect(":memory:")
    db.execute("CREATE TABLE sales (line_id INTEGER, category TEXT, quantity INTEGER, revenue REAL)")
    with open(path, encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        db.executemany("INSERT INTO sales VALUES (?, ?, ?, ?)", reader)
    started = time.perf_counter()
    db.execute("SELECT category, SUM(revenue), AVG(quantity) FROM sales GROUP BY category").fetchall()
    sqlite_time = time.perf_counter() - started
    db.close()

    frame = spark.read.csv(path, header=True, inferSchema=True).cache()
    frame.count()
    spark_time = float("inf")
    for attempt in range(3):
        started = time.perf_counter()
        frame.groupBy("category").agg(F.sum("revenue"), F.avg("quantity")).collect()
        spark_time = min(spark_time, time.perf_counter() - started)
    frame.unpersist()

    results.append((lines, sqlite_time, spark_time))
    print(f"{lines:>10,} rows   SQLite {sqlite_time:6.3f} s   Spark {spark_time:6.3f} s")

In [ ]:
print(f"{'rows':>10} {'SQLite':>9} {'Spark':>9}   faster")
for lines, sqlite_time, spark_time in results:
    if sqlite_time < spark_time:
        verdict = f"SQLite, {spark_time / sqlite_time:.0f} times"
    else:
        verdict = f"Spark, {sqlite_time / spark_time:.1f} times"
    print(f"{lines:>10,} {sqlite_time:>7.3f} s {spark_time:>7.3f} s   {verdict}")

Read the table from top to bottom. It is one of the most important results in the course.

**On small data, the single-machine database wins by a wide margin.** Spark's time hardly changes between the first row and the second, because almost none of it is spent on the data. It is spent on **overhead**: planning the query, scheduling tasks, starting them, collecting their results. That cost is the same for twenty thousand rows as for twenty.

**As the data grows, the positions reverse.** SQLite's time rises in step with the number of rows, since it does everything on one processor core. Spark divides the rows among all the cores it has, so its time rises far more slowly. Somewhere in this range the two lines cross: with many cores it happens near the second row, with only two it happens later. With more cores, or on a real cluster, Spark's advantage on large data becomes very large.

| | Thousands to a few hundred thousand rows | Millions of rows and beyond |
|---|---|---|
| A single-machine database | instant: no overhead | limited to one core, then to one machine's memory and disk |
| Spark | pays a fixed overhead on every query | spreads the work over every core and every machine |
| Better choice | **the single machine** | **Spark** |

Exactly where the lines cross depends on the machine, the query and the data. The shape does not. Every distributed system buys **scalability** with **overhead**. For small data you pay the overhead and get nothing for it.

Before choosing a tool, then, ask how big the data really is, and how big it will be in two years. Most datasets in most organisations are far smaller than the word "big" suggests, and fit comfortably in the memory of one modern machine. For those, the tools of Module A and Notebook 07 are simpler, cheaper, quicker to write, and faster. Spark is the right answer when the data has truly outgrown one machine, or soon will. Knowing which case you are in is worth more than knowing either tool.

## 6. Where we stand

This closes Module C. Its three steps, in order:

| The data no longer fits... | What we did |
|---|---|
| in memory | streamed it, a piece at a time, on one machine (Notebook 07) |
| in the time one machine has | split the work into pure functions over independent pieces (Notebook 08) |
| in a programmer's patience | described the work declaratively and let an engine plan it (Notebooks 09 and 10) |

The ideas kept returning under new names. The hash join of Notebook 02 came back as the broadcast join. The combiner of Notebook 08 came back as partial aggregation. Column pruning and predicate pushdown from Notebook 05 appeared, unasked, in Spark's plans. And the SQL of Notebook 01 turned out to need no changes at all. There are not many ideas in this field. There are a few good ones, applied at every scale.

One assumption has survived all ten notebooks. The data, however large, was **finished**: a file, a table, something with an end. The last module drops that assumption.

In [ ]:
spark.stop()
print("Spark stopped.")

---
## ✏️ Exercises

*The exercises need a running Spark session. If you ran the `spark.stop()` cell above, run the cell below first.*

In [ ]:
spark = (SparkSession.builder.master("local[*]").appName("bookshop")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
for name, path in paths.items():
    tables[name] = spark.read.csv(path, header=True, inferSchema=True)
    tables[name].createOrReplaceTempView(name)
print("Spark is running, with the tables", list(tables))

### Exercise 1 (SQL on Spark)
Write, in SQL, a query for the five cities with the highest revenue in 2024, and run it with `spark.sql`. You will need all four tables.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
spark.sql("""
    SELECT c.city, ROUND(SUM(i.quantity * p.price)) AS revenue
    FROM order_items AS i
    JOIN orders AS o    ON o.order_id = i.order_id
    JOIN customers AS c ON c.customer_id = o.customer_id
    JOIN products AS p  ON p.product_id = i.product_id
    WHERE YEAR(o.order_date) = 2024
    GROUP BY c.city
    ORDER BY revenue DESC
    LIMIT 5
""").show()
```

*It is the kind of query you wrote in Notebook 02. `YEAR(...)` works here because Spark recognised the column as a date when it read the file. In SQLite the date was text, and we cut the year out of it.*
</details>

### Exercise 2 (The same query with methods)
Rewrite the query of Exercise 1 with DataFrame methods (`join`, `filter`, `groupBy`, `agg`, `orderBy`, `limit`), starting from the DataFrames in the dictionary `tables`. Check that it gives the same rows.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
result = (tables["order_items"]
          .join(tables["orders"], "order_id")
          .join(tables["customers"], "customer_id")
          .join(tables["products"], "product_id")
          .filter(F.year("order_date") == 2024)
          .groupBy("city")
          .agg(F.round(F.sum(F.col("quantity") * F.col("price"))).alias("revenue"))
          .orderBy(F.desc("revenue"))
          .limit(5))
result.show()
```

*Joining on a column name given as a string, as here, keeps a single copy of that column in the result. Both versions produce the same plan.*
</details>

### Exercise 3 (Which join?)
Use the function `join_methods` from Section 2 to find out which kind of join Spark chooses for: (a) `order_items` with `products`, (b) `order_items` with `orders`. Then lower the broadcast limit to 100 kilobytes with `spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(100 * 1024))` and look again. Explain what changed. Set the limit back to `str(10 * 1024 * 1024)` afterwards.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
queries = {
    "items with products": "SELECT COUNT(*) FROM order_items AS i JOIN products AS p ON p.product_id = i.product_id",
    "items with orders":   "SELECT COUNT(*) FROM order_items AS i JOIN orders AS o ON o.order_id = i.order_id",
}
for limit in [10 * 1024 * 1024, 100 * 1024]:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(limit))
    print(f"Broadcast limit {limit // 1024:,} KB")
    for description, sql in queries.items():
        print(f"   {description:<22} {join_methods(spark.sql(sql))}")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
```

*With the default limit of 10 megabytes both joins are broadcast: even the orders file is only a few megabytes. With a limit of 100 kilobytes the products table still qualifies, and the orders table no longer does, so that join becomes a shuffle join. Spark decides by the estimated size of the smaller table. On a cluster, a table of a hundred megabytes is small, and the limit is often raised.*
</details>

### Exercise 4 (Replace a UDF)
The UDF below labels each product as `cheap`, `medium` or `expensive`. Rewrite it with Spark's built-in `F.when(...).when(...).otherwise(...)`, check that the two give the same counts, and time both on the joined table of order lines and prices.

In [ ]:
def band(price):
    if price < 20:
        return "cheap"
    if price < 40:
        return "medium"
    return "expensive"

band_udf = F.udf(band, StringType())
priced = tables["order_items"].join(F.broadcast(tables["products"]), "product_id").select("item_id", "price").cache()
priced.count()
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
band_built_in = F.when(F.col("price") < 20, "cheap").when(F.col("price") < 40, "medium").otherwise("expensive")

def run(expression):
    started = time.perf_counter()
    rows = priced.select(expression.alias("band")).groupBy("band").count().collect()
    return sorted(rows), time.perf_counter() - started

udf_rows, udf_time = run(band_udf("price"))
built_in_rows, built_in_time = run(band_built_in)

print(f"Python UDF: {udf_time:.2f} s    Built-in: {built_in_time:.2f} s")
print("Same counts:", udf_rows == built_in_rows)
priced.unpersist()
```

*A chain of conditions is the commonest reason people write a UDF, and `when` ... `otherwise` replaces it completely. It runs inside the engine, and the optimiser can see what it does.*
</details>

### Exercise 5 (Extend the pipeline, a little harder)
Add a stage to the pipeline of Section 4. Write `top_books(sales, how_many)` that returns, for each category, the `how_many` best-selling titles by revenue, with their rank. Use a window function through the DataFrame interface: `from pyspark.sql.window import Window`, then `F.rank().over(Window.partitionBy(...).orderBy(...))`. You will need to build `sales` again with `clean` and `enrich`. Add one check that the output is right.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from pyspark.sql.window import Window

def top_books(sales, how_many=3):
    totals = sales.groupBy("category", "title").agg(F.round(F.sum("revenue"), 2).alias("revenue"))
    ranked = totals.withColumn("position", F.rank().over(Window.partitionBy("category").orderBy(F.desc("revenue"))))
    return ranked.filter(F.col("position") <= how_many).orderBy("category", "position")

raw = spark.read.csv(raw_path, header=True, inferSchema=True)
sales = enrich(clean(raw), tables["products"], tables["orders"], tables["customers"]).cache()

best = top_books(sales, 2)
best.show(6)

categories = sales.select("category").distinct().count()
print("Check: at least 2 rows for each of the", categories, "categories:", best.count() >= 2 * categories)
sales.unpersist()
```

*The new stage is a function from a DataFrame to a DataFrame, like the others, so it slots into the pipeline without touching them. The check allows for more than two rows per category, since `rank` gives tied titles the same position.*
</details>

In [ ]:
spark.stop()

## 🔑 Key takeaways

- `spark.sql` runs ordinary **SQL** on DataFrames registered as views. SQL and DataFrame methods produce the **same plan**.
- Spark joins by **broadcasting** a small table to every worker, or by **shuffling** both tables on the key. Small dimension tables around a large fact table are the ideal case for broadcasting.
- A Python **UDF** carries every row out of the engine and back, and hides its logic from the optimiser. Prefer built-in functions.
- A **pipeline** is a chain of small stages: clean, enrich, summarise. Count what each stage rejects, verify what it produces, and make its output safe to write twice.
- Distributed engines trade **overhead** for **scalability**. On small data a single-machine database is faster. On large data the engine wins.
- The first question is always: **how big is the data, really?**

### 🏁 You have completed Module C: Too Big for One Machine!
You have processed files larger than memory on one machine, built a MapReduce engine and run jobs on it, and used a real cluster engine through RDDs, DataFrames and SQL. You have also measured when all of that is worth it.

**Next:** *Module D: Data in Motion and in the Cloud*, starting with *Notebook 11: Streaming*, where the data has no end, and an answer is never final.

---
*End of Module C.*

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*